# AMOCatlas demo

The purpose of this notebook is to demonstrate the functionality of `AMOCatlas`.

The demo is organised to show

- Step 1: Loading and plotting a sample dataset

- Step 2: Exploring the dataset attributes and variables.

Note that when you submit a pull request, you should `clear all outputs` from your python notebook for a cleaner merge.

In [ ]:
import pathlib
import sys
import os
from amocatlas import read, plotters

script_dir = pathlib.Path().parent.absolute()
parent_dir = script_dir.parents[0]
sys.path.append(str(parent_dir))

# Specify the path for writing datafiles
data_path = os.path.join(parent_dir, "data")

### Load RAPID 26°N

To see what files are available for each data source, use `read.DATASOURCE.list_files()`.  So for the RAPID array, you can use `read.rapid.list_files()`.

You can then use this list (or a subset thereof) to specify which files to be loaded and standardised using the `read.rapid(file_list=["file1", "file2"])` function with the input `file_list`.  It will return a list of xarray datasets, in the same order and the same length as `file_list`.  An exception is if `file_list` has length = 1, in which case it will return just the xarray dataset.

In [ ]:
# Find available files
rapid_file_list = read.rapid.list_files()

print("Available RAPID files:")
print(rapid_file_list)

# To specify to read two of those files, provide a file_list
standardRAPID = read.rapid(file_list = rapid_file_list[0:2])

In [ ]:
# Plot RAPID timeseries
plotters.plot_amoc_timeseries(
    data=[standardRAPID[0]],
    varnames=["MOC"],
    labels=[""],
    resample_monthly=True,
    plot_raw=True,
    figsize=(10, 5),
    title="RAPID 26°N"
)

### Load MOVE 16°N

To load the specified default transport file, use the `read.DATASOURCE()` function with no inputs.  So for the MOVE array, this is `read.move()`.  It will provide a single xarray dataset.  For each array, a default transport file has been specified.

*Developer's note:* This is specified in `~/amocatlas/data_source/DATASOURCE.py`.

In [ ]:
ds_move = read.move()

In [ ]:
# Plot MOVE timeseries
plotters.plot_amoc_timeseries(
    data=[ds_move],
    varnames=["MOC"],
    labels=[""],
    colors=["darkgreen"],
    resample_monthly=True,
    plot_raw=True,
    title="MOVE 16°N - NADW Transport"
)

### Load OSNAP

For just the transport file, you can use either `read.osnap()` as above, or `read.osnap(transport_only=True)`.  Both return a single xarray dataset.

In [ ]:
ds_osnap = read.osnap()
ds_osnap

In [ ]:
# Plot OSNAP timeseries
plotters.plot_amoc_timeseries(
    data=[ds_osnap],
    varnames=["MOC_SIGMA0"],
    labels=[""],
    colors=["darkblue"],
    resample_monthly=True,
    plot_raw=True,
    title="OSNAP"
)

### Load SAMBA 34.5°S

If you want to load all datasets available for a datasource, provide the option `all_files=True`.  Note that the first time you do this, it will download the data for you.  On subsequent runs (set up from the same location) it will use the previously downloaded data.

See the data reports in the docs: https://amoccommunity.github.io/AMOCatlas/ if you want to check how big a dataset is before downloading it.

In [ ]:
standardSAMBA = read.samba(all_files=True)


In [ ]:
# Plot SAMBA timeseries
plotters.plot_amoc_timeseries(
    data=[standardSAMBA[0], standardSAMBA[1]],
    varnames=["UPPER_TRANSPORT", "MOC"],
    labels=["Kersale et al. 2020", "Meinen et al. 2018"],
    colors=["grey", "blue"],
    title="SAMBA 34.5°S",
    time_limits=("2000-01-01", "2022-12-31"),
    ylim=(-25, 25),
    resample_monthly=True,
    plot_raw=False # Raw data is a little spiky
)

###  Load FW2015

For a formatted table showing the data including variables, dimensions, units etc, use the function `plotters.show_variables(DATASET)` where `DATASET` is an xarray dataset.

In [ ]:
standardfw2015 = read.fw2015()
plotters.show_variables(standardfw2015)

In [ ]:
# Plot timeseries
plotters.plot_amoc_timeseries(
    data=[standardfw2015],
    varnames=["MOC_PROXY"],
    labels=[""],
    colors=["darkblue"],
    resample_monthly=True,
    plot_raw=True,
    title="FW2015"
)

### LOAD MOCHA 26.5°N

Note that AMOCatlas renames variables and updates units according to defaults specified in the code.  For the MOCHA dataset, for example, the meridional heat transport is denoted by "Q", whereas for other datasets it is "MHT".  To simplify intercomparison, we rename heat transport variables to "MHT".  When you use the `read.mocha()` this variable remapping has already taken place.  

You can check the remapping applied in the docs: https://amoccommunity.github.io/AMOCatlas/

Additionally, some units such as "W" for Watts are converted to PetaWatts "PW" during the loading and standardisation.  If instead you want to load the raw data, you can use `read.mocha(raw=True)`.

In [ ]:
standardMOCHA = read.mocha()

plotters.show_variables(standardMOCHA)

In [ ]:
rawMOCHA = read.mocha(raw=True)
plotters.show_variables(rawMOCHA)

In [ ]:
# Plot timeseries
fig, ax = plotters.plot_amoc_timeseries(
    data=[standardMOCHA,standardMOCHA,standardMOCHA],
    varnames=["MHT", "MHT_OT","MHT_GYRE"],
    labels=["Total", "Overturning","Gyre"],
    colors=["red","darkblue","black"],
    resample_monthly=True,
    plot_raw=False,
    title="MOCHA"
)
ax.legend(loc="lower right")


### LOAD 41°N

Besides array-based datasets, some additional data sources are integrated within AMOCatlas.  For instance, the Willis and Hobbs estimates of heat transport at 41°N and the Willis transport estimates using Argo and altimetry are available as datasource "WH41N".  

In [ ]:
file_list = read.wh41n.list_files()
print("Available WH41N files:")
print(file_list)

standard41n = read.wh41n()

plotters.plot_amoc_timeseries(
    data=[standard41n],
    varnames=["MOC"],
    labels=[""],
    resample_monthly=True,
    plot_raw=False,
    colors=["darkblue"],
    title="41N"
)

### Load Denmark Strait overflow and Faroe Bank Channel overflow transports

Overflow transports for Denmark Strait and Faroe Bank Channel are also available.  

Note that in the current version of AMOCatlas we have not standardised the sign of the transports.  In this case, a stronger DSO is more negative, whereas a stronger FBC is more positive.

In [ ]:
standardDSO = read.dso()
standardFBC = read.fbc()

plotters.plot_amoc_timeseries(
    data=[standardDSO, standardFBC],
    varnames=["TRANS_DSO","TRANS_FBC"],
    labels=["DSO", "FBC"],
    resample_monthly=True,
    plot_raw=True,
    colors=["yellow", "orange"],
    title="DSO and FBC"
)


### Load Calafat2025

Meridional heat transport from a Bayesian method to produce a North Atlantic heat budget is also available.  However, it has 4000 realisations of the time series (from which uncertainties can be estimated), so is less straightforward to plot.

In [ ]:
standardCALAFAT2025 = read.calafat2025()

# declare latitude index (between 1 and 11)
lat_idx = 5
lat_val = standardCALAFAT2025['LATITUDE'].values[lat_idx]

if lat_val<0:
    title_str = f'CALAFAT2025 (lat = {-lat_val:.2f}°S)'
else:
    title_str = f'CALAFAT2025 (lat = {lat_val:.2f}°N)'

plotters.plot_amoc_timeseries(
data=[standardCALAFAT2025],
varnames=["MHT"],
labels=[""],
colors=["darkred"],
resample_monthly=True,
plot_raw=True,
lat_idx=lat_idx,
ylabel='MHT (PW)',
title=title_str
)

### Load Zheng2024

Freshwater transports also available.

In [ ]:
standardZHENG2024 = read.zheng2024()
# Declare latitude index (between 0 and 100)
lat_idx = 61
lat_val = standardZHENG2024['LATITUDE'].values[lat_idx]

# Set title string based on latitude value
if lat_val<0:
    title_str = f'ZHENG2024 (lat = {-lat_val:.2f}°S)'
else:
    title_str = f'ZHENG2024 (lat = {lat_val:.2f}°N)'

# Subselect data based on latitude index
data = standardZHENG2024['MFT'][:, lat_idx]

# Plot timeseries
plotters.plot_amoc_timeseries(
    data=[data],
    varnames=["MFT"],
    labels=[""],
    colors=["darkgreen"],
    resample_monthly=True,
    plot_raw=True,
    lat_idx=lat_idx,
    ylabel='MFT (Sv)',
    title=title_str
    )

### Monthly Anomalies Overview

In [ ]:
plotters.plot_monthly_anomalies(
    osnap_data=ds_osnap["MOC_SIGMA0"],
    fortyone_data = standard41n["MOC"],
    rapid_data=standardRAPID[0]["MOC"],
    move_data=-ds_move["MOC"],
    samba_data=standardSAMBA[1]["MOC"],
    fw2015_data=standardfw2015["MOC_PROXY"],
    dso_data = standardDSO["TRANS_DSO"],
    osnap_label="OSNAP",
    fortyone_label = "41°N",
    rapid_label="RAPID 26°N",
    move_label="MOVE 16°N",
    samba_label="SAMBA 34.5°S",
    fw2015_label="FW2015",
    dso_label = "DS Overflow Transport"
)

## Other components

It is also possible to manipulate (filter) and plot other components of the AMOC, depending on what is available in the datasets.

In [ ]:
clim = standardRAPID[0].groupby("TIME.month").mean("TIME")
tmp = standardRAPID[0].groupby("TIME.month") - clim
filtRAPID = tmp.rolling(TIME = 500, center = True).mean()

fig,ax = plotters.plot_amoc_timeseries(
    data=[filtRAPID],
    varnames=["TRANS_3000_5000"],
    labels=[""],
    resample_monthly=True,
    plot_raw=True,
    title="RAPID 26°N - t_ld10"
)
ax.set_ylim(4, -3)

fig.show()
